# Generating Text With A PretrainedLLM

In [1]:
import torch

print("Pytorch Version: ", torch.__version__)

Pytorch Version:  2.11.0+cu130


In [2]:
if torch.cuda.is_available():
    print(f"Cuda/ROCm GPU: {torch.cuda.get_device_name(0)}")
else:
    print("The device is running on cpu.!")

Cuda/ROCm GPU: NVIDIA L4


In [29]:
# %pip install git+https://github.com/rasbt/reasoning-from-scratch.git


## Preparing input text to for LLM

In [4]:
from reasoning_from_scratch.qwen3 import download_qwen3_small

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="Qwen3")


In [5]:
# Load the tokenizer

from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("Qwen3") / "tokenizer-base.json"

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

tokenizer_path


PosixPath('Qwen3/tokenizer-base.json')

In [6]:
tokenizer

In [7]:
# see it is working perfectly or not

prompt = "Explain the large language model"
input_token_ids_list = tokenizer.encode(prompt)
input_token_ids_list

[840, 20772, 279, 3460, 4128, 1614]

In [8]:
text = tokenizer.decode(input_token_ids_list)
text

'Explain the large language model'

In [9]:
for i in input_token_ids_list:
    print(f"{i} -----> {tokenizer.decode([i])}")

840 -----> Ex
20772 -----> plain
279 ----->  the
3460 ----->  large
4128 ----->  language
1614 ----->  model


### Get the device automatically

In [10]:
def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Device is using GPU")
        return device
    else:
        device=torch.device("cpu")
        print("Device is using CPU")
        return device

In [11]:
# to download model weight as well we can do with tokenizer_only=False

download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


In [12]:
device = get_device()

Device is using GPU


In [13]:
from reasoning_from_scratch.qwen3 import QWEN_CONFIG_06_B, Qwen3Model

model_path = Path("qwen3") / "qwen3-0.6B-base.pth"
model = Qwen3Model(QWEN_CONFIG_06_B) # Instantiate a Qwen3 model with random weight as placeholders

model.load_state_dict(torch.load(model_path)) # Load the pretrained weights into the model
model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [14]:
# lets see text generation in each iteration of the autoregressive process

prompt = "Explain large language models."

input_token_ids_list = tokenizer.encode(prompt)
print(f"Number of input tokens: {len(input_token_ids_list)}")

input_tensor = torch.tensor(input_token_ids_list) # convert python list into python sensor
input_tensor_fmt = input_tensor.unsqueeze(0) # Added an aditional dimension
input_tensor_fmt = input_tensor_fmt.to(device)

with torch.inference_mode():
    output_tensor = model(input_tensor_fmt) # generate the output

output_tensor_fmt = output_tensor.squeeze(0) # Removed the extra dimension
print("Formatted Output tensor shape: ", output_tensor_fmt.shape)

Number of input tokens: 6
Formatted Output tensor shape:  torch.Size([6, 151936])


In [15]:
# find the last position, because it predicts the next unseen token
last_token = output_tensor_fmt[-1]
print(last_token)

tensor([ 7.4062,  1.9609,  8.0000,  ..., -2.4219, -2.4219, -2.4219],
       device='cuda:0', dtype=torch.bfloat16)


In [16]:
# we will use argmax to find the position of the largest value score in the tensor

print(torch.argmax(last_token, dim=-1, keepdim=True))

tensor([20286], device='cuda:0')


In [17]:
# Lets see what is the text value

print(tokenizer.decode([20286]))

 Large


### Coding a minimal text generation function

Lets create a simple full text generation function

In [18]:
@torch.inference_mode() # it disable gradient tracking for speed and memory efficiently
def generate_text_basic_stream( 
        model,
        token_ids,
        max_new_tokens,
        eos_token_id = None
    ):
    model.eval() # switch model to evaluation mode to enable deterministic behavior(best practice)

    for _ in range(max_new_tokens):
        out = model(token_ids)[:, -1] # get the score of last token
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if(eos_token_id is not None and torch.all(next_token == eos_token_id)): # stop all the sequences in the batch have generated EOS
            break

        yield next_token # yield each token as soon as its generated

        token_ids = torch.cat([token_ids, next_token], dim=1) # append the newly predicted token to the sequence



In [19]:
# lets test the function for 100 max tokens

prompt = "Explain large language models in a single sentence"

input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device = device
).unsqueeze(0)
max_input_tokens =100

for token in generate_text_basic_stream(model = model, token_ids=input_token_ids_tensor, max_new_tokens=max_input_tokens):
    token_id = token.squeeze(0).tolist() # convert output token IDs from pytorch tensor to Python list)
    print(tokenizer.decode(token_id), 
        end="",
        flush=True) # deactivates buffering so tokens are printed live

.
Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as translation, summarization, and question answering.<|endoftext|>Human language is a complex and dynamic system that requires sophisticated models to understand and generate meaningful responses. Large language models are designed to mimic this complexity by learning from vast amounts of text data and using advanced algorithms to process and interpret language. These models can be trained on a wide range of sources, including books, articles, and

In [20]:
print(tokenizer.encode("<|endoftext|>"))

[151643]


In [21]:
# This token id(151643) is also saved via the tokenizer.eos_token_id

for token in generate_text_basic_stream(model = model, token_ids=input_token_ids_tensor, max_new_tokens=max_input_tokens, eos_token_id = tokenizer.eos_token_id):
    token_id = token.squeeze(0).tolist() # convert output token IDs from pytorch tensor to Python list)
    print(tokenizer.decode(token_id), 
        end="",
        flush=True) # deactivates buffering so tokens are printed live

# here will text generation stop when end_of_sequence apprear

.
Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as translation, summarization, and question answering.

#### Create a simple function that measures the run time

In [22]:
import warnings
def generate_stats(output_token_ids, tokenizer, start_time, end_time):
    total_time = end_time - start_time
    print(f"\n\nTime: {total_time:.2f} sec")
    print(f"{int(output_token_ids.numel() / total_time)} token/sec") #numel() stands for number of elements. If the model output 100 tokens, numel() gives 100

    for name, backend in (("CUDA", getattr(torch, "cuda", None)),
                        ("XPU", getattr(torch, "xpu", None))):
        
        if backend is not None and backend.is_available(): # Checks if an actual GPU card is physically detected on the machine.
            device_type = output_token_ids.device.type # Checks where the data actually was generated (either "cpu" or "cuda").
            if device_type != name.lower():
                warnings.warn(
                    f"{name} is available but tensor are on "
                    f"{device_type}. Memory states maybe 0."
                )
            if hasattr(backend, "synchronize"): # synchronize(): Finish all pending work first!
                backend.synchronize() 

            # The Fuel Gauge (Converting Bytes to Gigabytes)
            max_mem_bytes = backend.max_memory_allocated() # What was the highest amount of memory you used while generating this answer? GPU will answers in raw bytes.
            max_mem_gb = max_mem_bytes / (1024 ** 3) # Converting Bytes to Gigabytes
            print(f"Max {name} memory allocated: {max_mem_gb:.2f} GB")
            
            backend.reset_peak_memory_stats() # Resets the GPU's memory tracker back to zero so the next sentence starts with a clean slate.

In [23]:
import time
start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream(model = model, token_ids=input_token_ids_tensor, max_new_tokens=max_input_tokens, eos_token_id = tokenizer.eos_token_id):
    token_id = token.squeeze(0).tolist() # convert output token IDs from pytorch tensor to Python list)
    print(tokenizer.decode(token_id), 
        end="",
        flush=True) # deactivates buffering so tokens are printed live
    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)

end_time = time.time()
output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

.
Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as translation, summarization, and question answering.

Time: 1.46 sec
24 token/sec
Max CUDA memory allocated: 1.51 GB


### Faster inference via KV caching

In [24]:
# Basic text generation with KV caching

from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode() # it disable gradient tracking for speed and memory efficiently
def generate_text_basic_stream_cache( 
        model,
        token_ids,
        max_new_tokens,
        eos_token_id = None
    ):
    model.eval() # switch model to evaluation mode to enable deterministic behavior(best practice)
    cache = KVCache(n_layers=model.cfg["n_layers"]) # Initialize the KV cache
    model.reset_kv_cache()

    out = model(token_ids, cache=cache)[:, -1] #In the first round the whole input is provided to the model as before
    for _ in range(max_new_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if(eos_token_id is not None and torch.all(next_token == eos_token_id)): # stop all the sequences in the batch have generated EOS
            break

        yield next_token # yield each token as soon as its generated

        out = model(next_token, cache=cache)[:, -1] # consequent iterations only feed the next token to the input


In [25]:
# Lets see the speed and gpu allocation
start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream_cache(model = model, token_ids=input_token_ids_tensor, max_new_tokens=max_input_tokens, eos_token_id = tokenizer.eos_token_id):
    token_id = token.squeeze(0).tolist() # convert output token IDs from pytorch tensor to Python list)
    print(tokenizer.decode(token_id), 
        end="",
        flush=True) # deactivates buffering so tokens are printed live
    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

.
Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as translation, summarization, and question answering.

Time: 1.52 sec
23 token/sec
Max CUDA memory allocated: 1.45 GB


### Faster inference via Pytorch model compilation

In [26]:
major, minor = map(int, torch.__version__.split(".")[:2])

if (major, minor) >= (2, 8):
    # This avoid retriggering model recompilations
    # in pytorch 2.8 or newer
    # If the model contains code like self.ops = self.ops + 1
    torch._dynamo.config.allow_unspec_int_on_nn_module = True

model_compiled = torch.compile(model) # we can add extra parameter mode="max-autotune", if i could not see the improvement on eyes

In [27]:
# Generating text with the cimpiled model without KV cache

for i in range(3): # Run the token generation three times
    start_time = time.time()
    generated_ids = []

    for token in generate_text_basic_stream(model = model_compiled, token_ids=input_token_ids_tensor, max_new_tokens=max_input_tokens, eos_token_id = tokenizer.eos_token_id):
        token_id = token.squeeze(0).tolist() # convert output token IDs from pytorch tensor to Python list)
        print(tokenizer.decode(token_id), 
            end="",
            flush=True) # deactivates buffering so tokens are printed live
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)

    end_time = time.time()

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"/n/nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

    print(f"\n{30 * '-'}\n")
    

W1002 07:19:24.717000 1936 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


.
Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as translation, summarization, and question answering.

Warm-up run


Time: 101.13 sec
0 token/sec
Max CUDA memory allocated: 1.50 GB

------------------------------

.
Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as translation, summarization, and question answering./n/nTimed run 1:


Time: 0.54 sec
65 token/sec
Max CUDA memory allocated: 1.47 GB

------------------------------

.
Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as translation, summarization, and question answering./n/nTimed run 2:


Time: 0.51 sec
68 token/sec
Max CUDA memory allocated: 1.47 GB

------------------------------



In [28]:
# Generating text with the cimpiled model  KV cache

for i in range(3): # Run the token generation three times
    start_time = time.time()
    generated_ids = []

    for token in generate_text_basic_stream_cache(model = model_compiled, token_ids=input_token_ids_tensor, max_new_tokens=max_input_tokens, eos_token_id = tokenizer.eos_token_id):
        token_id = token.squeeze(0).tolist() # convert output token IDs from pytorch tensor to Python list)
        print(tokenizer.decode(token_id), 
            end="",
            flush=True) # deactivates buffering so tokens are printed live
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)

    end_time = time.time()

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"/n/nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

    print(f"\n{30 * '-'}\n")
    

.
Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as translation, summarization, and question answering.

Warm-up run


Time: 171.55 sec
0 token/sec
Max CUDA memory allocated: 1.50 GB

------------------------------

.
Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as translation, summarization, and question answering./n/nTimed run 1:


Time: 0.60 sec
58 token/sec
Max CUDA memory allocated: 1.46 GB

------------------------------

.
Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as translation, summarization, and question answering./n/nTimed run 2:


Time: 0.59 sec
59 token/sec
Max CUDA memory allocated: 1.46 GB

------------------------------

